In [1]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.express as px
import gcamreader

In [2]:
def ej_to_twh(ej):
    """
    Convert energy from exajoules (EJ) to terawatt-hours (TWh).

    Parameters:
    ej (float): Energy in exajoules.

    Returns:
    float: Energy in terawatt-hours.
    """
    twh = ej * 277.777778
    return twh

In [3]:
# =========================
# Config
# =========================
PROJECT_PATH = Path("/data/project/tae/gcam-core")
DB_REL_PATH  = "../output"
DB_FILE      = "database_basexdb_korea_2035_v5"
QUERY_FILE   = Path("..") / "output" / "queries" / "Main_queries.xml"

REGION       = "South Korea"
SCENARIOS    = ["Current-Policies-Med", "High-Ambition-Med"]
Q_GEN_TECH   = 9   # electricity generation by technology
YEARS_MARK   = list(range(2015, 2036, 5))  # for share markers/annotations

# Colors
custom_colors = {
    'Solar': '#FECB52',
    'Wind': 'rgb(136,204,238)',
    'Hydro': 'rgb(95, 70, 144)',
    'Nuclear': '#AB63FA',
    'Biomass': 'rgb(115, 175, 72)',
    'Gas w/ CCS': '#DEA0FD',
    'Gas': '#FFA15A',
    'Coal w/ CCS': '#750D86',
    'Coal': '#222A2A',
    'Oil': '#7D1215',
    'Hydrogen': "#727DCD",
    'Ammonia': "rgb(231,63,116)",
    'Others': 'rgb(217,217,217)',
}

stack_order = [
    'Ammonia', 'Hydrogen', 'Coal w/ CCS', 'Coal',
    'Gas w/ CCS', 'Gas', 'Oil', 'Nuclear',
    'Biomass', 'Hydro', 'Wind', 'Solar'
]

stack_order

['Ammonia',
 'Hydrogen',
 'Coal w/ CCS',
 'Coal',
 'Gas w/ CCS',
 'Gas',
 'Oil',
 'Nuclear',
 'Biomass',
 'Hydro',
 'Wind',
 'Solar']

In [4]:
# =========================
# Helpers
# =========================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def run_query(conn, q_idx):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[q_idx]
    df = conn.runQuery(q, scenarios=SCENARIOS, regions=[REGION])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def cat_tech(t):
    if t in ['PV', 'PV_storage', 'rooftoop_pv']: return 'Solar'
    if t in ['wind', 'wind_offshore', 'wind_storage']: return 'Wind'
    if t in ['hydro']: return 'Hydro'
    if t in ['Gen_III', 'Gen_II_LWR']: return 'Nuclear'
    if t in ['biomass (IGCC CCS)', 'biomass (conv CCS)']: return 'Biomass w/ CCS'
    if t in ['biomass (IGCC)', 'biomass (conv)']: return 'Biomass'
    if t in ['gas (CC CCS)']: return 'Gas w/ CCS'
    if t in ['gas (CC)', 'gas (steam/CT)']: return 'Gas'
    if t in ['coal (IGCC CCS)', 'coal (conv pul CCS)']: return 'Coal w/ CCS'
    if t in ['coal (IGCC)', 'coal (conv pul)']: return 'Coal'
    if t in ['refined liquids (CC CCS)']: return 'Oil w/ CCS'
    if t in ['refined liquids (CC)', 'refined liquids (steam/CT)']: return 'Oil'
    if t in ['gas (CC H2 blend 50%)']: return 'Hydrogen'
    if t in ['coal (conv pul ammonia blend 20%)']: return 'Ammonia'
    return 'Others'

def reallocate_h2_nh3(pivot: pd.DataFrame) -> pd.DataFrame:
    """
    Split Hydrogen & Ammonia to backing fuels (50% H2 → 50% Gas; 20% NH3 → 80% Coal)
    while preserving totals.
    """
    out = pivot.copy()
    # keep originals to compute remainders
    H_orig  = out.get('Hydrogen', pd.Series(0, index=out.index))
    NH3_orig= out.get('Ammonia',  pd.Series(0, index=out.index))

    out['Hydrogen'] = H_orig * 0.5
    out['Ammonia']  = NH3_orig * 0.2

    # Add the remaining to Gas / Coal
    out['Gas']  = out.get('Gas', 0)  + H_orig * 0.5
    out['Coal'] = out.get('Coal', 0) + NH3_orig * 0.8
    return out

def calc_shares(df_long: pd.DataFrame, scenario: str) -> tuple[pd.Series, pd.Series]:
    """Return RE% and Carbon-free% (RE + Nuclear) time series for a scenario."""
    is_re = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Hydrogen', 'Ammonia',])
    is_cf = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Hydrogen', 'Ammonia', 'Nuclear',])
    tot   = df_long[df_long['scenario'] == scenario].groupby('Year')['value'].sum()
    re    = df_long[(df_long['scenario'] == scenario) & is_re].groupby('Year')['value'].sum()
    cf    = df_long[(df_long['scenario'] == scenario) & is_cf].groupby('Year')['value'].sum()
    re_share = (re / tot * 100).reindex(YEARS_MARK)
    cf_share = (cf / tot * 100).reindex(YEARS_MARK)
    return re_share, cf_share

rank_map = {name: i for i, name in enumerate(stack_order)}
max_rank = len(stack_order) - 1

rank_map = {name: i for i, name in enumerate(stack_order)}
max_rank = len(stack_order) - 1

def add_stack_bars(fig, df_side: pd.DataFrame, col: int, show_legend: bool):
    cats = [c for c in stack_order if c in set(df_side["genTech"])]

    for c in cats:
        sub = df_side[df_side["genTech"] == c]
        fig.add_bar(
            name=c,
            x=sub["Year"],
            y=sub["value"],
            marker=dict(color=custom_colors.get(c)),
            showlegend=show_legend,
            legendrank=10 + (max_rank - rank_map[c]),  # ✅ legend 역순
            row=1, col=col, secondary_y=False
        )

In [5]:
conn = connect_db()

Database scenarios: High-Ambition-Med, Current-Policies-Med, High-Ambition-Med, Current-Policies-Med, High-Ambition-High, Current-Policies-High, High-Ambition-Med, Current-Policies-Med, Current-Policies-High, High-Ambition-Low, Current-Policies-Low, High-Ambition-Med-AI, Current-Policies-Med-AI, High-Ambition-Med-CPO2040, High-Ambition-Med, High-Ambition-Med


In [6]:
df = run_query(conn, 11)
df.head()

,Units,scenario,region,sector,subsector,technology,output,Year,value
0,EJ,Current-Policies-Med,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2020",elect_td_bld,2020,0.002046
1,EJ,Current-Policies-Med,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2025",elect_td_bld,2025,0.018703
2,EJ,Current-Policies-Med,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2030",elect_td_bld,2030,0.019130
3,EJ,Current-Policies-Med,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2035",elect_td_bld,2035,0.019182
4,EJ,Current-Policies-Med,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2025",elec_biomass (IGCC),2025,0.000015


In [7]:
df['tech'] = df['technology'].str.split(',').str[0]
df['tech'].unique()

array(['rooftop_pv', 'biomass (IGCC) (dry cooling)',
       'biomass (IGCC) (recirculating)', 'biomass (IGCC) (seawater)',
       'biomass (conv) (dry cooling)', 'biomass (conv) (once through)',
       'biomass (conv) (recirculating)', 'biomass (conv) (seawater)',
       'coal (IGCC CCS) (dry cooling)', 'coal (IGCC CCS) (recirculating)',
       'coal (IGCC CCS) (seawater)', 'coal (conv pul CCS) (dry cooling)',
       'coal (conv pul CCS) (once through)',
       'coal (conv pul CCS) (recirculating)',
       'coal (conv pul CCS) (seawater)',
       'coal (conv pul ammonia blend 20%)',
       'coal (conv pul) (dry cooling)', 'coal (conv pul) (once through)',
       'coal (conv pul) (recirculating)', 'coal (conv pul) (seawater)',
       'gas (CC CCS) (dry cooling)', 'gas (CC CCS) (recirculating)',
       'gas (CC CCS) (seawater)', 'gas (CC H2 blend 50%)',
       'gas (CC) (dry cooling)', 'gas (CC) (once through)',
       'gas (CC) (recirculating)', 'gas (CC) (seawater)',
       'gas (steam

In [8]:
# 1) Load generation by technology
df = run_query(conn, Q_GEN_TECH)
df['genTech'] = df['technology'].map(cat_tech)
df = df[~df['genTech'].isna()].copy()

# 2) Convert EJ to TWh
df['value'] = df['value'].apply(ej_to_twh)
df['Units'] = 'TWh'

In [9]:
df.head()

,Units,scenario,region,subsector,technology,output,Year,value,genTech
0,TWh,Current-Policies-Med,South Korea,biomass,biomass (IGCC),electricity,2025,0.187946,Biomass
1,TWh,Current-Policies-Med,South Korea,biomass,biomass (IGCC),electricity,2030,0.607097,Biomass
2,TWh,Current-Policies-Med,South Korea,biomass,biomass (IGCC),electricity,2035,1.166139,Biomass
3,TWh,Current-Policies-Med,South Korea,biomass,biomass (conv),electricity,2005,0.045001,Biomass
4,TWh,Current-Policies-Med,South Korea,biomass,biomass (conv),electricity,2010,0.345997,Biomass


In [10]:
df['subsector'].unique()

array(['biomass', 'coal', 'gas', 'hydro', 'nuclear', 'refined liquids',
       'rooftop_pv', 'solar', 'wind'], dtype=object)

In [11]:
df['technology'].unique()

array(['biomass (IGCC)', 'biomass (conv)', 'coal (IGCC CCS)',
       'coal (conv pul CCS)', 'coal (conv pul ammonia blend 20%)',
       'coal (conv pul)', 'gas (CC CCS)', 'gas (CC H2 blend 50%)',
       'gas (CC)', 'gas (steam/CT)', 'hydro', 'Gen_III', 'Gen_II_LWR',
       'refined liquids (CC)', 'refined liquids (steam/CT)', 'rooftop_pv',
       'PV', 'PV_storage', 'wind', 'wind_offshore', 'wind_storage'],
      dtype=object)

In [12]:
def rec(tech):
    if tech == 'wind_offshore':
        return 2
    if tech == 'wind_storage':
        return 2
    if tech in ['biomass (IGCC)', 'biomass (conv)']:
        return 1
    if tech == 'PV_storage':
        return 2
    if tech in ['wind', 'PV', 'hydro']:
        return 1

In [13]:
df.head()


,Units,scenario,region,subsector,technology,output,Year,value,genTech
0,TWh,Current-Policies-Med,South Korea,biomass,biomass (IGCC),electricity,2025,0.187946,Biomass
1,TWh,Current-Policies-Med,South Korea,biomass,biomass (IGCC),electricity,2030,0.607097,Biomass
2,TWh,Current-Policies-Med,South Korea,biomass,biomass (IGCC),electricity,2035,1.166139,Biomass
3,TWh,Current-Policies-Med,South Korea,biomass,biomass (conv),electricity,2005,0.045001,Biomass
4,TWh,Current-Policies-Med,South Korea,biomass,biomass (conv),electricity,2010,0.345997,Biomass


In [14]:
df['rec_coef'] = df['technology'].apply(rec)
df['rec'] = df['value'] * df['rec_coef'] 

In [15]:
df[(df['rec_coef'] == 1.5)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech,rec_coef,rec


In [16]:
df[(df['rec_coef'].isna())]

,Units,scenario,region,subsector,technology,output,Year,value,genTech,rec_coef,rec
10,TWh,Current-Policies-Med,South Korea,coal,coal (IGCC CCS),electricity,2025,1.509667,Coal w/ CCS,NaN,NaN
11,TWh,Current-Policies-Med,South Korea,coal,coal (IGCC CCS),electricity,2030,1.243936,Coal w/ CCS,NaN,NaN
12,TWh,Current-Policies-Med,South Korea,coal,coal (IGCC CCS),electricity,2035,1.249242,Coal w/ CCS,NaN,NaN
13,TWh,Current-Policies-Med,South Korea,coal,coal (conv pul CCS),electricity,2025,1.235067,Coal w/ CCS,NaN,NaN
14,TWh,Current-Policies-Med,South Korea,coal,coal (conv pul CCS),electricity,2030,0.772628,Coal w/ CCS,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
186,TWh,High-Ambition-Med,South Korea,refined liquids,refined liquids (steam/CT),electricity,2035,5.465889,Oil,NaN,NaN
187,TWh,High-Ambition-Med,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2020,0.568292,Others,NaN,NaN
188,TWh,High-Ambition-Med,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2025,5.195333,Others,NaN,NaN
189,TWh,High-Ambition-Med,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2030,5.854250,Others,NaN,NaN


In [17]:
listTechRe = ['wind_offshore', 'wind_storage', 'PV', 'biomass (IGCC)', 'biomass (conv)', 'hydro']
(df[(df['technology'].isin(listTechRe))].groupby(['Year', 'scenario'])['rec'].sum() / df[(df['technology'] != 'rooftop_pv')].groupby(['Year', 'scenario'])['value'].sum())

Year  scenario            
1990  Current-Policies-Med    0.063950
      High-Ambition-Med       0.063950
2005  Current-Policies-Med    0.010229
      High-Ambition-Med       0.010229
2010  Current-Policies-Med    0.010187
      High-Ambition-Med       0.010187
2015  Current-Policies-Med    0.017133
      High-Ambition-Med       0.017133
2020  Current-Policies-Med    0.051571
      High-Ambition-Med       0.051571
2025  Current-Policies-Med    0.097522
      High-Ambition-Med       0.097522
2030  Current-Policies-Med    0.235834
      High-Ambition-Med       0.356900
2035  Current-Policies-Med    0.348005
      High-Ambition-Med       0.553345
dtype: float64

In [18]:
0.11 / 0.89

0.12359550561797752

In [19]:
0.235 / 0.765

0.3071895424836601

In [20]:
0.27 / 0.73

0.36986301369863017

In [21]:
0.25 * 0.95

0.2375

In [22]:
0.36 * 0.95

0.34199999999999997

In [23]:
listTechRe = ['wind_offshore', 'wind_storage', 'PV', 'biomass (IGCC)', 'biomass (conv)', 'hydro']
df[(df['technology'].isin(listTechRe))].groupby(['Year', 'scenario'])['value'].sum() / df[(~df['technology'].isin(['rooftop_pv']))].groupby(['Year', 'scenario'])['value'].sum()

Year  scenario            
1990  Current-Policies-Med    0.063950
      High-Ambition-Med       0.063950
2005  Current-Policies-Med    0.010229
      High-Ambition-Med       0.010229
2010  Current-Policies-Med    0.010187
      High-Ambition-Med       0.010187
2015  Current-Policies-Med    0.017133
      High-Ambition-Med       0.017133
2020  Current-Policies-Med    0.051523
      High-Ambition-Med       0.051523
2025  Current-Policies-Med    0.094542
      High-Ambition-Med       0.094542
2030  Current-Policies-Med    0.189007
      High-Ambition-Med       0.251058
2035  Current-Policies-Med    0.263128
      High-Ambition-Med       0.380514
Name: value, dtype: float64

In [24]:
df.groupby(['Year', 'scenario'])['value'].sum()

Year  scenario            
1990  Current-Policies-Med     99.483878
      High-Ambition-Med        99.483878
2005  Current-Policies-Med    364.947533
      High-Ambition-Med       364.947533
2010  Current-Policies-Med    471.185461
      High-Ambition-Med       471.185461
2015  Current-Policies-Med    521.740750
      High-Ambition-Med       521.740750
2020  Current-Policies-Med    537.753683
      High-Ambition-Med       537.753683
2025  Current-Policies-Med    593.229512
      High-Ambition-Med       593.229512
2030  Current-Policies-Med    656.844166
      High-Ambition-Med       689.021452
2035  Current-Policies-Med    692.272031
      High-Ambition-Med       770.485528
Name: value, dtype: float64

In [25]:
dfFig = (
    df[(df['Year'] >= 2015) & (df['Year'] <= 2035)]
    .groupby(['scenario', 'Year', 'genTech'], observed=False)['value']
    .sum().reset_index()
)
dfFig

,scenario,Year,genTech,value
0,Current-Policies-Med,2015,Biomass,2.324597
1,Current-Policies-Med,2015,Coal,215.851667
2,Current-Policies-Med,2015,Gas,120.367295
3,Current-Policies-Med,2015,Hydro,2.642000
4,Current-Policies-Med,2015,Nuclear,164.662778
...,...,...,...,...
101,High-Ambition-Med,2035,Nuclear,236.111111
102,High-Ambition-Med,2035,Oil,12.890556
103,High-Ambition-Med,2035,Others,11.380028
104,High-Ambition-Med,2035,Solar,173.437832


In [26]:
pivot = dfFig.pivot_table(index=['scenario', 'Year'], columns='genTech', values='value', fill_value=0)
pivot

genTech                      Ammonia    Biomass        Coal  Coal w/ CCS  \
scenario             Year                                                  
Current-Policies-Med 2015   0.000000   2.324597  215.851667     0.000000   
                     2020   0.000000   2.290144  196.944722     0.000000   
                     2025   0.000000   3.047140  153.906945     2.744733   
                     2030  41.111111   4.345569   77.366111     2.016564   
                     2035  87.222222   5.815444   18.963250     2.010339   
High-Ambition-Med    2015   0.000000   2.324597  215.851667     0.000000   
                     2020   0.000000   2.290144  196.944722     0.000000   
                     2025   0.000000   3.047140  153.906945     2.744733   
                     2030   0.000000   5.233608   83.566667     8.333361   
                     2035   0.000000  10.625067    0.215551    17.500000   

genTech                           Gas  Gas w/ CCS     Hydro   Hydrogen  \
scenario             Year                                                
Current-Policies-Med 2015  120.367295    0.000000  2.642000   0.000000   
                     2020  132.446161    0.000000  4.722222   0.000000   
                     2025  141.202436    2.152547  3.611111   0.000000   
                     2030  125.065558    1.313808  3.611111   9.291861   
                     2035   67.965701    1.475064  3.611111  14.186000   
High-Ambition-Med    2015  120.367295    0.000000  2.642000   0.000000   
                     2020  132.446161    0.000000  4.722222   0.000000   
                     2025  141.202436    2.152547  3.611111   0.000000   
                     2030  127.963883    4.166639  3.611111   5.927083   
                     2035   79.367561    8.333333  3.611111  15.001000   

genTech                       Nuclear        Oil     Others       Solar  \
scenario             Year                                                 
Current-Policies-Med 2015  164.662778  10.578611   0.000000    3.972611   
                     2020  160.277833  12.104056   0.568292   20.639963   
                     2025  180.917500  16.818611   5.195333   47.203665   
                     2030  204.166667  10.628306   5.313778   84.785019   
                     2035  234.163334   8.580250   5.328278  113.374447   
High-Ambition-Med    2015  164.662778  10.578611   0.000000    3.972611   
                     2020  160.277833  12.104056   0.568292   20.639963   
                     2025  180.917500  16.818611   5.195333   47.203665   
                     2030  204.166667  11.161389   5.854250  105.382276   
                     2035  236.111111  12.890556  11.380028  173.437832   

genTech                          Wind  
scenario             Year              
Current-Policies-Med 2015    1.341192  
                     2020    7.760289  
                     2025   36.429489  
                     2030   87.828702  
                     2035  129.576591  
High-Ambition-Med    2015    1.341192  
                     2020    7.760289  
                     2025   36.429489  
                     2030  123.654517  
                     2035  202.012378

In [27]:
out = pivot.copy()
H_orig  = out.get('Hydrogen', pd.Series(0, index=out.index))
NH3_orig= out.get('Ammonia',  pd.Series(0, index=out.index))

out['Hydrogen'] = H_orig * 0.5
out['Ammonia']  = NH3_orig * 0.2

out['Gas']  = out.get('Gas', 0)  + H_orig * 0.5
out['Coal'] = out.get('Coal', 0) + NH3_orig * 0.8

In [28]:
result_df = (
    out
    .reset_index()
    .melt(id_vars=['scenario', 'Year'], var_name='genTech', value_name='value')
)

# Order & clean
result_df['genTech'] = pd.Categorical(result_df['genTech'], categories=stack_order, ordered=True)
result_df = result_df.sort_values(['scenario', 'Year', 'genTech'])

# 4) Shares
re_current, cf_current = calc_shares(result_df, SCENARIOS[0])
re_enh,    cf_enh      = calc_shares(result_df, SCENARIOS[1])

In [36]:
dfFig.groupby(['scenario', 'Year'])['value'].sum()

scenario              Year
Current-Policies-Med  2015    521.740750
                      2020    537.753683
                      2025    593.229512
                      2030    656.844166
                      2035    692.272031
High-Ambition-Med     2015    521.740750
                      2020    537.753683
                      2025    593.229512
                      2030    689.021452
                      2035    770.485528
Name: value, dtype: float64

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    shared_xaxes=True, shared_yaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policies", "High Ambition"),
    column_widths=[0.5, 0.5],
    horizontal_spacing=0.08
)

# ----------------------------
# Bars
# ----------------------------
left  = result_df[result_df['scenario'] == SCENARIOS[0]]
right = result_df[result_df['scenario'] == SCENARIOS[1]]

add_stack_bars(fig, left,  col=1, show_legend=True)
add_stack_bars(fig, right, col=2, show_legend=False)

# ----------------------------
# Share markers (REAL) — legend off
# ----------------------------
fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=re_current, mode="markers",
    name="Renewable Share",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    showlegend=False
), row=1, col=1, secondary_y=True)

fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=cf_current, mode="markers",
    name="Carbon-Free Share",
    marker=dict(symbol='triangle-up', size=9, color="blue"),
    showlegend=False
), row=1, col=1, secondary_y=True)

fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=re_enh, mode="markers",
    name="Renewable Share",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    showlegend=False
), row=1, col=2, secondary_y=True)

fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=cf_enh, mode="markers",
    name="Carbon-Free Share",
    marker=dict(symbol='triangle-up', size=9, color="blue"),
    showlegend=False
), row=1, col=2, secondary_y=True)

# ----------------------------
# Annotate % values near markers
# ----------------------------
for year in YEARS_MARK:
    if pd.notna(re_current.get(year)):
        fig.add_annotation(
            x=year, y=re_current.get(year) + 3, text=f"{re_current.get(year):.0f}%",
            xref="x1", yref="y2", showarrow=False, font=dict(color="green")
        )
    if pd.notna(cf_current.get(year)):
        fig.add_annotation(
            x=year, y=cf_current.get(year) + 3, text=f"{cf_current.get(year):.0f}%",
            xref="x1", yref="y2", showarrow=False, font=dict(color="blue")
        )
    if pd.notna(re_enh.get(year)):
        fig.add_annotation(
            x=year, y=re_enh.get(year) + 3, text=f"{re_enh.get(year):.0f}%",
            xref="x2", yref="y4", showarrow=False, font=dict(color="green")
        )
    if pd.notna(cf_enh.get(year)):
        fig.add_annotation(
            x=year, y=cf_enh.get(year) + 3, text=f"{cf_enh.get(year):.0f}%",
            xref="x2", yref="y4", showarrow=False, font=dict(color="blue")
        )

# ----------------------------
# Dummy legend entries (Share only) + spacing line
#   - legendrank를 크게 줘서 발전원 아래로 보내기
#   - 공백 trace 하나로 간격 만들기
# ----------------------------
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(size=0, opacity=0),
    name=" ",  # spacing line
    showlegend=True,
    hoverinfo="skip",
    legendrank=1999
))



fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="blue"),
    name="Carbon-Free Share",
    showlegend=True,
    legendrank=2001
))


fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    name="Renewable Share",
    showlegend=True,
    legendrank=2003
))

# ----------------------------
# Layout / axes
# ----------------------------
fig.update_layout(
    width=850,
    height=700,
    barmode="stack",
    plot_bgcolor="rgba(0,0,0,0)",
    legend=dict(
        traceorder="normal",   # legendrank 적용 정렬
        x=1.02, y=1,
        font=dict(size=15),
        borderwidth=0
    ),
    margin=dict(r=220)  # legend 오른쪽 공간 확보 (필요시 조정)
)

# Primary y (left title only)
fig.update_yaxes(
    title_text="Electricity Generation (TWh)", range=[0, 810],
    row=1, col=1, secondary_y=False,
    showgrid=True, gridcolor="lightgray"
)
fig.update_yaxes(
    title_text=None,range=[0, 810],
    row=1, col=2, secondary_y=False,
    showgrid=True, gridcolor="lightgray"
)

# Secondary y-axes (share %): hide ticks, no grid, fixed range
fig.update_yaxes(
    secondary_y=True, range=[0, 100],
    showticklabels=False, title_text=None,
    showgrid=False,
    row=1, col=1
)
fig.update_yaxes(
    secondary_y=True, range=[0, 100],
    showticklabels=False, title_text=None,
    showgrid=False,
    row=1, col=2
)

# X axes
for c in (1, 2):
    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(2015, 2040, 5)),
        tickangle=45,
        tickfont=dict(size=15),
        row=1, col=c
    )

fig.update_layout(
    legend=dict(
        traceorder="normal",  # ✅ legendrank 정렬 유지
        font=dict(size=15),
        x=1.02, y=1,
        borderwidth=0
    )
)
# Fonts
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_annotations(font=dict(size=21))

pio.write_image(fig, "./fig/power.jpg", width=850, height=700, scale=3)
fig.show()


In [30]:
result_df

,scenario,Year,genTech,value
0,Current-Policies-Med,2015,Ammonia,0.000000
70,Current-Policies-Med,2015,Hydrogen,0.000000
30,Current-Policies-Med,2015,Coal w/ CCS,0.000000
20,Current-Policies-Med,2015,Coal,215.851667
50,Current-Policies-Med,2015,Gas w/ CCS,0.000000
...,...,...,...,...
19,High-Ambition-Med,2035,Biomass,10.625067
69,High-Ambition-Med,2035,Hydro,3.611111
129,High-Ambition-Med,2035,Wind,202.012378
119,High-Ambition-Med,2035,Solar,173.437832


In [31]:
def calc_shares(df_long: pd.DataFrame, scenario: str) -> tuple[pd.Series, pd.Series]:
    """Return RE% and Carbon-free% (RE + Nuclear) time series for a scenario."""
    is_re = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Hydrogen', 'Ammonia', 'Others'])
    is_cf = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Hydrogen', 'Ammonia', 'Nuclear', 'Others'])
    tot   = df_long[df_long['scenario'] == scenario].groupby('Year')['value'].sum()
    re    = df_long[(df_long['scenario'] == scenario) & is_re].groupby('Year')['value'].sum()
    cf    = df_long[(df_long['scenario'] == scenario) & is_cf].groupby('Year')['value'].sum()
    re_share = (re / tot * 100).reindex(YEARS_MARK)
    cf_share = (cf / tot * 100).reindex(YEARS_MARK)
    return re_share, cf_share

In [32]:
df['genTech'] = df['technology'].apply(cat_tech)
df[(df['Year'] == 2030)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech,rec_coef,rec
1,TWh,Current-Policies-Med,South Korea,biomass,biomass (IGCC),electricity,2030,0.607097,Biomass,1.0,0.607097
8,TWh,Current-Policies-Med,South Korea,biomass,biomass (conv),electricity,2030,3.738472,Biomass,1.0,3.738472
11,TWh,Current-Policies-Med,South Korea,coal,coal (IGCC CCS),electricity,2030,1.243936,Coal w/ CCS,NaN,NaN
14,TWh,Current-Policies-Med,South Korea,coal,coal (conv pul CCS),electricity,2030,0.772628,Coal w/ CCS,NaN,NaN
16,TWh,Current-Policies-Med,South Korea,coal,coal (conv pul ammonia blend 20%),electricity,2030,41.111111,Ammonia,NaN,NaN
24,TWh,Current-Policies-Med,South Korea,coal,coal (conv pul),electricity,2030,77.366111,Coal,NaN,NaN
27,TWh,Current-Policies-Med,South Korea,gas,gas (CC CCS),electricity,2030,1.313808,Gas w/ CCS,NaN,NaN
29,TWh,Current-Policies-Med,South Korea,gas,gas (CC H2 blend 50%),electricity,2030,9.291861,Hydrogen,NaN,NaN
37,TWh,Current-Policies-Med,South Korea,gas,gas (CC),electricity,2030,124.706111,Gas,NaN,NaN
45,TWh,Current-Policies-Med,South Korea,gas,gas (steam/CT),electricity,2030,0.359447,Gas,NaN,NaN
